# Problema 2: Tarifas de un hotel

*Enunciado adaptado del problema 8 del Banco de Problemas de Aprendizaje por Refuerzo (MDT-Módulo 3, IIND-3518). Al final del cuaderno se explica qué se modificó.*


**Enunciado:** Un hotel escoge cada día una tarifa baja ($b$) o alta ($a$) según su ocupación: baja ($B$), media ($M$) o alta ($A$). El ingreso del día depende de la ocupación y de la tarifa:
| Ocupación | Tarifa baja ($b$) | Tarifa alta ($a$) |
|---|---|---|
| B | 6,5 | 3 |
| M | 6 | 4 |
| A | 7 | 10 |

La ocupación del día siguiente es **aleatoria**: desde una misma ocupación y con la misma tarifa, el hotel puede amanecer al día siguiente con ocupación baja, media o alta. Las probabilidades dependen solo de la ocupación de hoy y de la tarifa elegida, y están dadas por las siguientes tablas. Cada fila corresponde a la ocupación de hoy, cada columna a la ocupación de mañana, y cada fila suma 1.

**Si se cobra la tarifa baja ($b$):**

| Ocupación hoy | Mañana B | Mañana M | Mañana A |
|---|---|---|---|
| B | 0,3 | 0,5 | 0,2 |
| M | 0,2 | 0,4 | 0,4 |
| A | 0,1 | 0,3 | 0,6 |

**Si se cobra la tarifa alta ($a$):**

| Ocupación hoy | Mañana B | Mañana M | Mañana A |
|---|---|---|---|
| B | 0,7 | 0,3 | 0 |
| M | 0,4 | 0,4 | 0,2 |
| A | 0,2 | 0,5 | 0,3 |

Por ejemplo, si hoy la ocupación es baja y se cobra la tarifa baja, mañana la ocupación será baja con probabilidad 0,3, media con probabilidad 0,5 y alta con probabilidad 0,2. Con ocupación baja y tarifa alta, en cambio, la ocupación no puede subir a alta al día siguiente.

El algoritmo de Q-learning **no conoce estas probabilidades**: solo observa, en cada día simulado, la ocupación a la que efectivamente se pasa y el ingreso recibido.

El hotel opera indefinidamente, empieza con ocupación baja y usa $\gamma = 0{,}9$.

- (a) Formule el problema para Q-learning: estados, acciones factibles y qué observa el algoritmo después de cada decisión.
- (b) Entrene Q-learning con $\alpha = 0{,}01$, $\varepsilon = 0{,}2$, $2 \times 10^5$ pasos y semilla 2026. Presente la tabla $Q$ y la política greedy aprendida.
- (c) Calcule la política óptima de forma exacta con *value iteration* y compárela con la aprendida en (b). Compare también la ganancia máxima de cada estado con la ganancia de seguir la política aprendida.
- (d) Interprete la política óptima y compárela con la política greedy $(b, b, a)$ que se obtiene en el problema original después de procesar las seis experiencias.

---
## Preparación: importar la librería

In [1]:
import sys
import os.path
import numpy as np
# context for the examples
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..', '..')))
from jmarkov.mdp.qlearning import qlearning
from jmarkov.mdp.dtmdp import dtmdp

---
## Solución (a)

**Aquí se usa la librería:** el problema se describe con las tres funciones que recibe la clase `qlearning`.

- **Estados:** la ocupación del día, $s \in \{B, M, A\}$. La función `initial_state` devuelve $B$.
- **Acciones factibles:** las dos tarifas, $b$ y $a$, en todos los estados. Lo expresa la función `feasible_actions`.
- **Qué observa el algoritmo:** la función `step` simula un día. Sortea la ocupación del día siguiente con la fila de la tabla de transiciones y devuelve **(ocupación siguiente, ingreso)**. Q-learning aprende de esa observación; nunca recibe las probabilidades.

In [ ]:
GAMMA = 0.9
OCUPACIONES = ["B", "M", "A"]
TARIFAS = ["b", "a"]

INGRESO = {
    "B": {"b": 6.5, "a": 3.0},
    "M": {"b": 6.0, "a": 4.0},
    "A": {"b": 7.0, "a": 10.0},
}

# PROB[tarifa][ocupación actual] = probabilidades de pasar a B, M y A.
PROB = {
    "b": {"B": [0.3, 0.5, 0.2], "M": [0.2, 0.4, 0.4], "A": [0.1, 0.3, 0.6]},
    "a": {"B": [0.7, 0.3, 0.0], "M": [0.4, 0.4, 0.2], "A": [0.2, 0.5, 0.3]},
}


def initial_state(rng):
    return "B"  # el hotel empieza con ocupación baja pero podría empezar con la misma probabilidad así:
#def initial_state(rng):
#    return OCUPACIONES[int(rng.integers(3))]   # B, M o A con la misma probabilidad


def feasible_actions(state):
    return ("b", "a")


def step(state, action, rng):
    siguiente = OCUPACIONES[int(rng.choice(3, p=PROB[action][state]))]
    return siguiente, INGRESO[state][action]


model = qlearning(initial_state, feasible_actions, step, discount_factor=GAMMA)   # <- LIBRERÍA

---
## Solución (b)

**Aquí se usa la librería:** `model.solve(...)` entrena Q-learning durante $2 \times 10^5$ pasos, siguiendo una sola trayectoria desde la ocupación baja y eligiendo las tarifas con $\varepsilon$-greedy. Devuelve la función de valor estimada, $\hat V(s) = \max_a Q(s,a)$, y la política greedy. La tabla $Q$ completa queda en `model.Q`.

In [4]:
valores_ql, politica_ql = model.solve(n_steps=200_000, alpha=0.01, epsilon=0.2, minimize=False, seed=2026)   # <- LIBRERÍA

print("Tabla Q aprendida:")
for s in OCUPACIONES:
    print(f"  {s}:   Q({s},b) = {model.Q[s]['b']:7.3f}   Q({s},a) = {model.Q[s]['a']:7.3f}")

print("\nPolítica greedy aprendida:", {s: politica_ql[s] for s in OCUPACIONES})
print("Valores estimados:        ", {s: round(valores_ql[s], 3) for s in OCUPACIONES})

Tabla Q aprendida:
  B:   Q(B,b) =  72.501   Q(B,a) =  68.490
  M:   Q(M,b) =  72.911   Q(M,a) =  70.037
  A:   Q(A,b) =  74.461   Q(A,a) =  76.625

Política greedy aprendida: {'B': 'b', 'M': 'b', 'A': 'a'}
Valores estimados:         {'B': 72.501, 'M': 72.911, 'A': 76.625}


---
## Solución (c)

**Aquí se usa jMarkov, pero no la clase de Q-learning:** la solución exacta se obtiene con la clase `dtmdp`, que aplica *value iteration*. A diferencia de Q-learning, `dtmdp` recibe el modelo completo: una matriz de transición por tarifa (las tablas del enunciado) y la matriz de ingresos (filas: ocupaciones; columnas: tarifas).

In [5]:
estados = np.array(OCUPACIONES)
acciones = np.array(TARIFAS)
matrices = {t: np.array([PROB[t][s] for s in OCUPACIONES]) for t in TARIFAS}
recompensas = np.array([[INGRESO[s][t] for t in TARIFAS] for s in OCUPACIONES])

mdp = dtmdp(estados, acciones, matrices, recompensas, GAMMA)                                   
valores_exactos, politica_exacta = mdp.solve(1e-12, minimize=False, method="value_iteration")   
print(f"{'Ocupación':>9} {'Política exacta':>16} {'Política Q-learning':>20}")
for s in OCUPACIONES:
    print(f"{s:>9} {str(politica_exacta[s]):>16} {politica_ql[s]:>20}")

Ocupación  Política exacta  Política Q-learning
        B                b                    b
        M                b                    b
        A                a                    a


**Ganancia de seguir la política aprendida.** Se calcula de forma exacta resolviendo $(I - \gamma P_\pi)V = r_\pi$, donde $P_\pi$ y $r_\pi$ son las probabilidades de transición y las recompensas esperadas de la acción que eligió Q-learning en cada estado. Si la política aprendida es óptima, su ganancia es igual a la ganancia máxima.

In [6]:
P_pi = np.array([matrices[politica_ql[s]][i] for i, s in enumerate(OCUPACIONES)])
r_pi = np.array([INGRESO[s][politica_ql[s]] for s in OCUPACIONES])
ganancia_ql = np.linalg.solve(np.eye(len(OCUPACIONES)) - GAMMA * P_pi, r_pi)

print(f"{'Ocupación':>9} {'Ganancia máxima':>16} {'Ganancia política Q-learning':>29}")
for i, s in enumerate(OCUPACIONES):
    print(f"{s:>9} {valores_exactos[i]:>16.3f} {ganancia_ql[i]:>29.3f}")

Ocupación  Ganancia máxima  Ganancia política Q-learning
        B           72.717                        72.717
        M           72.893                        72.893
        A           76.563                        76.563


Las dos políticas coinciden en las tres ocupaciones, y por eso la ganancia de seguir la política aprendida es igual a la ganancia máxima.

---
## Solución (d)

**Aquí no se usa la librería:** es la interpretación de la política obtenida en (b) y (c).

- **Ocupación baja → tarifa baja.** Deja más ingreso hoy (6,5 frente a 3) y además tiende a subir la ocupación.
- **Ocupación media → tarifa baja.** También deja más ingreso hoy (6 frente a 4) y aumenta la probabilidad de pasar a ocupación alta.
- **Ocupación alta → tarifa alta.** Deja el mayor ingreso del día (10 frente a 7), aunque aumente la probabilidad de que la ocupación baje.

La política óptima, $(b, b, a)$, es la misma que la política greedy del problema original después de las seis experiencias. Sin embargo, allí se obtuvo con seis actualizaciones a partir de una tabla inicial arbitraria; aquí se confirma con un entrenamiento largo y con la solución exacta del modelo completo.

---
## Nota sobre la adaptación

El problema 8 original da una tabla $Q$ inicial y seis experiencias, y pide procesarlas a mano. Para pedir la política óptima con la librería, el hotel necesita un modelo completo, que el enunciado original no trae. Por eso:

- **Ingresos:** se tomaron de las experiencias del enunciado original. La pareja $(B, b)$ aparece dos veces, con ingresos 8 y 5, y se usa su promedio, 6,5. El ingreso de $(B, a)$, que no aparece en el original, se fijó en 3.
- **Transiciones:** el original no da probabilidades, así que se definieron con un criterio: la tarifa baja tiende a subir la ocupación y la alta a bajarla. Todas las transiciones observadas en el problema original tienen probabilidad positiva.
- **Horizonte:** el hotel opera indefinidamente, así que se eliminó el estado terminal $T$. Se conserva $\gamma = 0{,}9$.